In [ ]:
import numpy as np
from scipy.integrate import quad
import matplotlib as plt

'''
log-loss
    objective function to minimize (and converge upon optimal Tau value for Glicko-2)
brier score as benchmark to validate model signaling over guessing 
    different objective function (which handles residual-lengths less harshly than log-loss and reports overfitting)
    non_arbitrary number serves as benchmark
AUC ranking quality
    quantifies if rankings are credible based on binary random sampling
ECE + reliability curve
    diagnostic tool for seeing at what confidence intervals the model is overconfident and underconfident
'''

def log_loss(outcomes, probabilities):
    nudge = 1e-10
    probabilities = np.clip(probabilities, nudge, 1 - nudge) #prevents log converging to literal infinity by slight nudge away from 0
    return -1 * np.mean(outcomes * np.log(probabilities) + (1 - outcomes) * np.log(1 - probabilities))

def brier_skill_score(outcomes, probabilities):
    baseline_pred_rate = 0.5
    naive_model = np.mean((baseline_pred_rate - outcomes) ** 2)
    brier_model = np.mean((probabilities - outcomes) ** 2)
    return 1 - (brier_model / naive_model)

def ranking_calibration(outcomes, probabilities):

    actual = np.array(outcomes).astype(bool) #boolean array
    probabilities = np.array(probabilities) # for elementwise operations
    tpr_points = []
    fpr_points = []

    for cutoff in np.linspace(1, 0, 101):

        pred = np.array(probabilities > cutoff)
        true_pos = np.sum(pred & actual) #bitwise operators of booleans, ~means flip bool value
        false_pos = np.sum(pred & ~actual)
        true_neg = np.sum(~pred & ~actual)
        false_neg = np.sum(~pred & actual)

        true_pos_rate = true_pos / (true_pos + false_pos)
        false_pos_rate = true_neg / (true_neg + false_neg)

        tpr_points.append(true_pos_rate)
        fpr_points.append(false_pos_rate)

    area_under_curve = np.trapezoid(tpr_points, fpr_points)
    #AI generated plotting code; could not be bothered
    plt.figure(figsize=(6, 6))
    plt.plot(fpr_points, tpr_points, label=f'Model (AUC = {auc:.3f})')
    plt.plot([0, 1], [0, 1], '--', color='gray', label='Guessing (AUC = 0.5)')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend(loc='lower right')
    plt.gca().set_aspect('equal')
    plt.show()

    return area_under_curve

def confidence_calibration(outcomes, probabilities, step):
    probabilities = np.array(probabilities)
    outcomes = np.array(outcomes)

    confidence_bins = np.array[np.arange(0 , 100 - step, step), np.arange(0 + step, 100, step)]
    residuals_points = np.array()

    for i, bin in enumerate(confidence_bins):
        if bin[0] <=  probabilities < bin[1]:
            residuals_points.append(True)
        else:
            residuals_points.append(False)

def full_benchmark(outcomes, probabilities, step):
    print(log_loss(outcomes, probabilities))
    print(brier_skill_score(outcomes, probabilities))
    print(ranking_calibration(outcomes, probabilities))
    print(confidence_calibration(outcomes, probabilities, step))
#